# Рекомендация тарифов

В вашем распоряжении данные о поведении клиентов, которые уже перешли на эти тарифы (из проекта курса «Статистический анализ данных»). Нужно построить модель для задачи классификации, которая выберет подходящий тариф. Предобработка данных не понадобится — вы её уже сделали.

Постройте модель с максимально большим значением *accuracy*. Чтобы сдать проект успешно, нужно довести долю правильных ответов по крайней мере до 0.75. Проверьте *accuracy* на тестовой выборке самостоятельно.

## Откроем файл и изучим его

Импортируем необходимые для работы библиотеки. Считаем данные из csv-файла в датафрейм и сохраним в переменную df. Выводим на экран первые 10 строк csv-файла, изучим сводную информацию о таблице.

In [1]:
# импорт библиотек
import pandas as pd
from sklearn.model_selection import train_test_split 
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from joblib import dump
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score


In [2]:
# чтение csv-файли и сохранение его в переменную df
try:
    df = pd.read_csv('/datasets/users_behavior.csv')
except:
    df = pd.read_csv('C:/Users/evgen/Documents/data/Проекты по аналитике Яндекс/Данные для проектов/Тарифы/users_behavior.csv')

In [3]:
# просмотр сводной информации о таблице
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [4]:
# проверим наличие в таблице явных дубликатов
df.duplicated().sum()

0

В таблице 3214 строк и 5 столбцов, пропусков в столбцах нет. Тип данных в столбцах c количеством звонков и сообщений можно изменить на int, т.к. это целочисленные значения. Явные дубликаты в таблице не обнаружены.

In [5]:
# изменим тип данных в столбцах calls, messages на int
df[['calls', 'messages']] = df[['calls', 'messages']].astype('Int64', errors='raise')

## Разбъем данные на выборки

Для дальнейшего исследования разобьем данные на обучающую(60%), валидационную(20%) и тестовую выборки(20%). После разделения тестовую выборку временно "отложим".

In [6]:
# разобьем данные на обучающую и валидационную выборки
df_train, df_val = train_test_split(df, test_size=0.40, random_state=123, stratify=df['is_ultra'])

In [7]:
# отделим от валидационных данных тестовую выборку
df_valid, df_test = train_test_split(df_val, test_size=0.50, random_state=123)

In [8]:
# создадим переменные для признаков и целевого признака для обучающей выборки
features_train = df_train.drop(['is_ultra'], axis=1)
target_train = df_train['is_ultra']

In [9]:
# создадим переменные для признаков и целевого признака для валидационной выборки
features_valid = df_valid.drop(['is_ultra'], axis=1)
target_valid = df_valid['is_ultra']

In [10]:
# создадим переменные для признаков и целевого признака для тестовой выборки
features_test = df_test.drop(['is_ultra'], axis=1)
target_test = df_test['is_ultra']

In [11]:
print('Размер обучающей выборки:', (df_train.shape[0]/df.shape[0])*100)
print('Размер валидационной выборки:', (df_valid.shape[0]/df.shape[0])*100)
print('Размер тестовой выборки:', (df_test.shape[0]/df.shape[0])*100)


Размер обучающей выборки: 59.98755444928439
Размер валидационной выборки: 20.00622277535781
Размер тестовой выборки: 20.00622277535781


## Исследуем модели

### Модель Дерево решений

In [12]:
# создадим и обучим модель на обучающей выборке
model_tree = DecisionTreeClassifier(random_state=123)
model_tree.fit(features_train, target_train)

DecisionTreeClassifier(random_state=123)

In [13]:
# проверим параметр accuracy на обучающей и валидационной выборках
train_predictions = model_tree.predict(features_train) 
valid_predictions = model_tree.predict(features_valid) 

print("Обучающая выборка:", accuracy_score(target_train, train_predictions)) 
print("Валидационная выборка:", accuracy_score(target_valid, valid_predictions))

Обучающая выборка: 1.0
Валидационная выборка: 0.6967340590979783


In [14]:
# подберем значение гиперпараметра max_depth для дерева решений
for max_depth in range(1, 16):
    model_tree = DecisionTreeClassifier(random_state=123, max_depth=max_depth) 
    model_tree.fit(features_train, target_train)

    predictions_valid = model_tree.predict(features_valid)

    print("max_depth =", max_depth, ": ", end='')
    print(accuracy_score(target_valid, predictions_valid))

max_depth = 1 : 0.7527216174183515
max_depth = 2 : 0.7776049766718507
max_depth = 3 : 0.7853810264385692
max_depth = 4 : 0.7838258164852255
max_depth = 5 : 0.7853810264385692
max_depth = 6 : 0.7853810264385692
max_depth = 7 : 0.7853810264385692
max_depth = 8 : 0.7916018662519441
max_depth = 9 : 0.76049766718507
max_depth = 10 : 0.7744945567651633
max_depth = 11 : 0.7698289269051322
max_depth = 12 : 0.7542768273716952
max_depth = 13 : 0.744945567651633
max_depth = 14 : 0.7465007776049767
max_depth = 15 : 0.7325038880248833


Лучшие значения accuracy при глубине дерева 8, при величине параметра max_depth от 10 и выше значение accuracy начинает снижаться.

### Модель Случайный лес

In [15]:
best_model = None
best_result = 0
trees_count = 0
depth = 0
for est in range(1, 16):
    model_forest = RandomForestClassifier(random_state=123, n_estimators=est, max_depth=10) 
    model_forest.fit(features_train, target_train)
    result = model_forest.score(features_valid, target_valid) 
    if result > best_result:
        best_model = model_forest
        trees_count = est
        depth = model_forest.estimators_[0].tree_.max_depth
        best_result = result 

print("Accuracy наилучшей модели на валидационной выборке:", best_result)
print(f"Количество деревьев: {trees_count}")
print(f"Глубина: {depth}")

Accuracy наилучшей модели на валидационной выборке: 0.8040435458786936
Количество деревьев: 10
Глубина: 10


Лучшие значение Accuracy модели Случайный лес равно 0,804 при количестве деревьев - 10 и глубине дерева 10. Это значение немного выше, чем у дерева решений - 0,791.

### Модель Логистическая регрессия

In [16]:
model_logistic = LogisticRegression(random_state=123, solver='lbfgs', max_iter=100)
model_logistic.fit(features_train, target_train)
model_logistic.predict(features_valid) 
model_logistic.score(features_valid, target_valid) 


0.6998444790046656

Наилучшие значения Accuracy для разных моделей:
- дерево решений 0,791
- случайный лес 0,804
- логистическая регрессия 0,699

Самое высокое качество у модели Случайный лес, следом за ним идет Дерево решений.

## Проверим модель на тестовой выборке

In [17]:
model_forest.predict(features_test) 
accuracy_test = model_forest.score(features_test, target_test) 
print("Accuracy модели Случайный лес на тестовой выборке:", round(accuracy_test, 3))


Accuracy модели Случайный лес на тестовой выборке: 0.82


## (бонус) Проверим модели на адекватность

In [18]:
# Считаем количество клиентов тарифа "смарт"
smart_count = df[df['is_ultra'] == 0]['is_ultra'].count()

# Считаем общее количество клиентов
total_count = df['is_ultra'].count()

# Рассчитываем точность базовой модели
base_accuracy = smart_count / total_count
print("Точность базовой модели:", base_accuracy)


Точность базовой модели: 0.693528313627878


In [19]:
accuracies_test = cross_val_score(model_forest, features_test, target_test, scoring='accuracy', cv=10)
print("Средняя точность модели по всем циклам кросс-валидации:", accuracies_test.mean())

Средняя точность модели по всем циклам кросс-валидации: 0.8072836538461539


Точность базовой модели (когда всем клиентам назначен тариф "Смарт") равно 0,693. Это значение ниже Accuracy модели, посчитанной на тестовой и валидационной выборках.  Средняя точность модели по всем циклам кросс-валидации составляет около 0,807. Это означает, что модель достаточно хорошо обобщает данные и может быть использована для прогнозирования значений целевой переменной.

## Общий вывод

В ходе работы было проанализировано поведение клиентов с целью предложения пользователям нового тарифа: «Смарт» или «Ультра». Были построены 3 модели: 'дерево решений', 'случайный лес', 'логистическая регрессия'. По результатам сравнения была выбрана лучшая - Случайный лес с гиперпараметрами n_estimators = 10 и max_depth = 10, accuracy = 0.804.

Лучшая модель проверена на тестовых объектах, не участвующих в обучении и валидации модели. Accuracy модели Случайный лес на тестовой выборке: 0.82. 

Точность базовой модели (когда всем клиентам назначен тариф "Смарт") равно 0,693. Это значение ниже Accuracy модели, посчитанной на тестовой и валидационной выборках. Средняя точность лучшей модели Случайный лес по всем циклам кросс-валидации составляет около 0,807. 

Рекомендуется для построения системы анализа поведения клиентов использовать модель Случайный лес, так как она имеет наибольшую точность предсказания значения целевой переменной.